# Baseline BFP-PE and BucketGetter Full-Trace Verification

This standalone notebook reconstructs the mathematical G16/BFP4 result from the shared LLaMA2-7B `input.dat`, verifies the Baseline cycle output, and measures BucketGetter's final-result error. BucketGetter uses four 128-bit circular buckets with four exponent levels per bucket.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RELATIVE_EPSILON = 1e-12
REPORT_RELATIVE_TOLERANCE = 1e-6
SAVE_CYCLE_REPORT = False


def find_rtl_root() -> Path:
    current = Path.cwd().resolve()
    candidates = []
    for root in (current, *current.parents):
        candidates.append(root / 'research1' / 'rtl')
        candidates.append(root / 'rtl')
        candidates.append(root)
    for candidate in dict.fromkeys(candidates):
        if (candidate / '01_Baseline-BFP-PE').is_dir() and (candidate / '02_Bucket-Getter-PE').is_dir():
            return candidate
    raise FileNotFoundError('Cannot locate the RTL project root from the current working directory.')


RTL_ROOT = find_rtl_root()
BASELINE_TESTBED = RTL_ROOT / '01_Baseline-BFP-PE' / '00_TESTBED'
BUCKET_TESTBED = RTL_ROOT / '02_Bucket-Getter-PE' / '00_TESTBED'
VERIFICATION_DIR = RTL_ROOT / '00_verification'
REPORT_DIR = VERIFICATION_DIR / 'reports'
INPUT_PATH = BASELINE_TESTBED / 'input.dat'
BASELINE_OUTPUT_PATH = BASELINE_TESTBED / 'output.dat'
BUCKET_OUTPUT_PATH = BUCKET_TESTBED / 'output.dat'
METADATA_PATH = VERIFICATION_DIR / 'trace_metadata.json'
INDEX_PATH = VERIFICATION_DIR / 'trace_index.csv'

required_paths = [INPUT_PATH, BASELINE_OUTPUT_PATH, BUCKET_OUTPUT_PATH, METADATA_PATH, INDEX_PATH]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('Missing verification files:\n' + '\n'.join(missing_paths))

metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
trace_index = pd.read_csv(INDEX_PATH)
assert metadata['trace_format'] == 'baseline_bfp_pe_cycle_input_v2'
assert metadata['shared_exponent_semantics'] == (
    'value = signed_integer_magnitude * 2**(encoded_exponent - exponent_bias)'
)
assert metadata['bfp_config']['group_size'] == 16
assert metadata['bfp_config']['mantissa_bits'] == 3
assert metadata['bfp_config']['exponent_bias'] == 15

GROUP_SIZE = int(metadata['bfp_config']['group_size'])
MANTISSA_BITS = int(metadata['bfp_config']['mantissa_bits'])
EXPONENT_BIAS = int(metadata['bfp_config']['exponent_bias'])
TOTAL_CYCLES = int(metadata['total_cycles'])
CYCLES_PER_DOT = int(metadata['cycles_per_dot_product'])
DOT_PRODUCT_COUNT = int(metadata['dot_product_count'])
BUCKET_CONFIG = {
    'bucket_count': 4,
    'bucket_width': 128,
    'exponent_levels_per_bucket': 4,
}

print({
    'rtl_root': str(RTL_ROOT),
    'trace_format': metadata['trace_format'],
    'total_cycles': TOTAL_CYCLES,
    'dot_products': DOT_PRODUCT_COUNT,
    'bucket_config': BUCKET_CONFIG,
})

## Parse and validate the shared trace

Baseline writes one word per scheduled input cycle. BucketGetter writes one drained FP32 word per dot product, in `trace_index.csv` order.

In [ ]:
FIELD_NAMES = [
    'acc_clear', 'weight_load', 'in_valid',
    'w_sign', 'w_exp', 'w_magnitude',
    'a_sign', 'a_exp', 'a_magnitude',
]
FIELD_PATTERNS = {
    'acc_clear': r'[01]',
    'weight_load': r'[01]',
    'in_valid': r'[01]',
    'w_sign': r'[0-9a-fA-F]{4}',
    'w_exp': r'[0-9a-fA-F]{2}',
    'w_magnitude': r'[0-9a-fA-F]{12}',
    'a_sign': r'[0-9a-fA-F]{4}',
    'a_exp': r'[0-9a-fA-F]{2}',
    'a_magnitude': r'[0-9a-fA-F]{12}',
}

stimulus_hex = pd.read_csv(INPUT_PATH, sep=r'\s+', names=FIELD_NAMES, dtype=str, engine='python')
baseline_output_hex = pd.read_csv(
    BASELINE_OUTPUT_PATH, header=None, names=['word'], dtype=str
)['word'].str.strip()
bucket_output_hex = pd.read_csv(
    BUCKET_OUTPUT_PATH, header=None, names=['word'], dtype=str
)['word'].str.strip()

assert len(stimulus_hex) == TOTAL_CYCLES
assert len(baseline_output_hex) == TOTAL_CYCLES
assert len(bucket_output_hex) == DOT_PRODUCT_COUNT
assert len(trace_index) == DOT_PRODUCT_COUNT
for field, pattern in FIELD_PATTERNS.items():
    if not stimulus_hex[field].str.fullmatch(pattern).all():
        raise ValueError(f'Malformed {field} field in input.dat')
for name, words in [('Baseline', baseline_output_hex), ('BucketGetter', bucket_output_hex)]:
    if not words.str.fullmatch(r'[0-9a-fA-F]{8}').all():
        raise ValueError(f'Malformed 32-bit word in {name} output.dat')

stimulus = stimulus_hex.copy()
for field in FIELD_NAMES:
    stimulus[field] = stimulus[field].map(lambda value: int(value, 16))

position = np.arange(TOTAL_CYCLES, dtype=np.int64) % CYCLES_PER_DOT
np.testing.assert_array_equal(stimulus['acc_clear'].to_numpy(dtype=bool), position == 0)
np.testing.assert_array_equal(stimulus['weight_load'].to_numpy(dtype=bool), position < CYCLES_PER_DOT-1)
np.testing.assert_array_equal(stimulus['in_valid'].to_numpy(dtype=bool), position != 0)
expected_setup = np.arange(DOT_PRODUCT_COUNT, dtype=np.int64) * CYCLES_PER_DOT
np.testing.assert_array_equal(trace_index['setup_cycle'].to_numpy(dtype=np.int64), expected_setup)
np.testing.assert_array_equal(trace_index['final_cycle'].to_numpy(dtype=np.int64), expected_setup + CYCLES_PER_DOT-1)

print(f'[PASS] Parsed {TOTAL_CYCLES:,} input cycles, {len(baseline_output_hex):,} Baseline outputs, and {len(bucket_output_hex)} BucketGetter results.')

## Reconstruct the mathematical G16/BFP4 reference

The reference follows the trace's integer-magnitude convention: each block contributes `integer_dot * 2**(w_exp + a_exp - 2*15)`.

In [ ]:
def unpack_sign(word: int) -> np.ndarray:
    return np.fromiter(
        ((word >> lane) & 1 for lane in range(GROUP_SIZE)),
        dtype=np.int8,
        count=GROUP_SIZE,
    )


def unpack_magnitude(word: int) -> np.ndarray:
    mask = (1 << MANTISSA_BITS) - 1
    return np.fromiter(
        ((word >> (lane * MANTISSA_BITS)) & mask for lane in range(GROUP_SIZE)),
        dtype=np.int16,
        count=GROUP_SIZE,
    )


def decode_fp32_words(words: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    bits = np.fromiter((int(word, 16) for word in words), dtype=np.uint32, count=len(words))
    values = bits.astype('>u4', copy=False).view('>f4').astype(np.float64)
    return bits, values


reference_cycle = np.zeros(TOTAL_CYCLES, dtype=np.float64)
integer_dot = np.zeros(TOTAL_CYCLES, dtype=np.int32)
block_value = np.zeros(TOTAL_CYCLES, dtype=np.float64)
weight_sign_reg = 0
weight_exp_reg = 0
weight_magnitude_reg = 0
accumulator = 0.0

for cycle, row in enumerate(stimulus.itertuples(index=False, name=None)):
    (
        acc_clear, weight_load, in_valid,
        w_sign, w_exp, w_magnitude,
        a_sign, a_exp, a_magnitude,
    ) = row
    if acc_clear:
        accumulator = 0.0
    elif in_valid:
        products = unpack_magnitude(weight_magnitude_reg) * unpack_magnitude(a_magnitude)
        product_sign = unpack_sign(weight_sign_reg) ^ unpack_sign(a_sign)
        dot = int(np.where(product_sign != 0, -products, products).sum(dtype=np.int32))
        value = np.ldexp(float(dot), int(weight_exp_reg) + int(a_exp) - 2*EXPONENT_BIAS)
        accumulator += value
        integer_dot[cycle] = dot
        block_value[cycle] = value
    reference_cycle[cycle] = accumulator
    if weight_load:
        weight_sign_reg = int(w_sign)
        weight_exp_reg = int(w_exp)
        weight_magnitude_reg = int(w_magnitude)

baseline_bits, baseline_cycle = decode_fp32_words(baseline_output_hex)
bucket_bits, bucket_final = decode_fp32_words(bucket_output_hex)
final_cycles = trace_index['final_cycle'].to_numpy(dtype=np.int64)
reference_final = reference_cycle[final_cycles]
baseline_final = baseline_cycle[final_cycles]
print({
    'reference_range': [float(reference_final.min()), float(reference_final.max())],
    'baseline_range': [float(baseline_final.min()), float(baseline_final.max())],
    'bucketgetter_range': [float(bucket_final.min()), float(bucket_final.max())],
})

## Compare final results and export reports

In [ ]:
def error_metrics(actual: np.ndarray, reference: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    absolute = np.abs(actual - reference)
    relative = absolute / np.maximum(np.abs(reference), RELATIVE_EPSILON)
    return absolute, relative


baseline_cycle_abs, baseline_cycle_rel = error_metrics(baseline_cycle, reference_cycle)
baseline_abs, baseline_rel = error_metrics(baseline_final, reference_final)
bucket_abs, bucket_rel = error_metrics(bucket_final, reference_final)
bucket_vs_baseline_abs, bucket_vs_baseline_rel = error_metrics(bucket_final, baseline_final)

comparison_report = trace_index.copy()
comparison_report['reference'] = reference_final
comparison_report['baseline_rtl_hex'] = baseline_output_hex.iloc[final_cycles].to_numpy()
comparison_report['baseline_rtl'] = baseline_final
comparison_report['baseline_absolute_error'] = baseline_abs
comparison_report['baseline_relative_error'] = baseline_rel
comparison_report['bucketgetter_rtl_hex'] = bucket_output_hex.to_numpy()
comparison_report['bucketgetter_rtl'] = bucket_final
comparison_report['bucketgetter_absolute_error'] = bucket_abs
comparison_report['bucketgetter_relative_error'] = bucket_rel
comparison_report['bucketgetter_vs_baseline_absolute_difference'] = bucket_vs_baseline_abs
comparison_report['bucketgetter_vs_baseline_relative_difference'] = bucket_vs_baseline_rel
comparison_report['bucketgetter_within_reporting_tolerance'] = bucket_rel <= REPORT_RELATIVE_TOLERANCE

summary = {
    'trace_format': metadata['trace_format'],
    'total_cycles': TOTAL_CYCLES,
    'dot_product_count': DOT_PRODUCT_COUNT,
    'report_relative_tolerance': REPORT_RELATIVE_TOLERANCE,
    'bucketgetter_config': BUCKET_CONFIG,
    'baseline': {
        'exact_final_matches': int(np.count_nonzero(baseline_bits[final_cycles] == np.fromiter((int(v, 16) for v in baseline_output_hex.iloc[final_cycles]), dtype=np.uint32))),
        'zero_error_final_results': int(np.count_nonzero(baseline_abs == 0.0)),
        'max_final_relative_error': float(baseline_rel.max()),
        'max_cycle_relative_error': float(baseline_cycle_rel.max()),
    },
    'bucketgetter': {
        'bit_exact_matches_vs_baseline': int(np.count_nonzero(bucket_bits == baseline_bits[final_cycles])),
        'zero_error_final_results_vs_reference': int(np.count_nonzero(bucket_abs == 0.0)),
        'within_reporting_tolerance': int(np.count_nonzero(bucket_rel <= REPORT_RELATIVE_TOLERANCE)),
        'max_absolute_error': float(bucket_abs.max()),
        'max_relative_error': float(bucket_rel.max()),
        'mean_relative_error': float(bucket_rel.mean()),
        'median_relative_error': float(np.median(bucket_rel)),
        'p95_relative_error': float(np.percentile(bucket_rel, 95)),
    },
}

REPORT_DIR.mkdir(parents=True, exist_ok=True)
comparison_report.to_csv(REPORT_DIR / 'final_dot_product_comparison.csv', index=False)
if SAVE_CYCLE_REPORT:
    pd.DataFrame({
        'cycle': np.arange(TOTAL_CYCLES, dtype=np.int64),
        'reference': reference_cycle,
        'baseline_rtl': baseline_cycle,
        'absolute_error': baseline_cycle_abs,
        'relative_error': baseline_cycle_rel,
    }).to_csv(REPORT_DIR / 'baseline_cycle_error_report.csv', index=False)
(REPORT_DIR / 'verification_summary.json').write_text(
    json.dumps(summary, indent=2) + '\n', encoding='utf-8'
)

print(json.dumps(summary, indent=2))
print('\nWorst BucketGetter results:')
print(comparison_report.nlargest(10, 'bucketgetter_relative_error').to_string(index=False))

## Visualize Baseline and BucketGetter error

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(16, 11))

axes[0, 0].scatter(reference_final, baseline_final, s=22, alpha=0.65, label='Baseline')
axes[0, 0].scatter(reference_final, bucket_final, s=14, alpha=0.65, marker='x', label='BucketGetter')
limits = [min(reference_final.min(), baseline_final.min(), bucket_final.min()), max(reference_final.max(), baseline_final.max(), bucket_final.max())]
axes[0, 0].plot(limits, limits, 'k--', linewidth=1)
axes[0, 0].set_title('Final result vs mathematical reference')
axes[0, 0].set_xlabel('Reference')
axes[0, 0].set_ylabel('RTL output')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

x = np.arange(DOT_PRODUCT_COUNT)
axes[0, 1].semilogy(x, np.maximum(baseline_rel, 1e-18), label='Baseline')
axes[0, 1].semilogy(x, np.maximum(bucket_rel, 1e-18), label='BucketGetter')
axes[0, 1].axhline(REPORT_RELATIVE_TOLERANCE, color='r', linestyle='--', label='Reporting tolerance')
axes[0, 1].set_title('Final relative error')
axes[0, 1].set_xlabel('Dot-product index')
axes[0, 1].set_ylabel('Relative error')
axes[0, 1].grid(True, which='both', alpha=0.3)
axes[0, 1].legend()

axes[1, 0].plot(x, bucket_final - reference_final)
axes[1, 0].axhline(0.0, color='k', linewidth=1)
axes[1, 0].set_title('BucketGetter signed error')
axes[1, 0].set_xlabel('Dot-product index')
axes[1, 0].set_ylabel('RTL - reference')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(bucket_rel, bins=30, alpha=0.8)
axes[1, 1].set_title('BucketGetter relative-error distribution')
axes[1, 1].set_xlabel('Relative error')
axes[1, 1].set_ylabel('Count')
axes[1, 1].grid(True, alpha=0.3)

figure.tight_layout()
figure.savefig(REPORT_DIR / 'verification_plots.png', dpi=160, bbox_inches='tight')
plt.show()

## Final status

Baseline remains a strict correctness guard. BucketGetter is reported numerically because its approximation error depends on bucket configuration and trace distribution.

In [ ]:
if not np.all(baseline_rel <= REPORT_RELATIVE_TOLERANCE):
    raise AssertionError('Baseline BFP-PE numerical verification failed.')

print(f'[PASS] Baseline: {np.count_nonzero(baseline_rel <= REPORT_RELATIVE_TOLERANCE)}/{DOT_PRODUCT_COUNT} within {REPORT_RELATIVE_TOLERANCE:.1e}.')
print(
    f'[INFO] BucketGetter: {np.count_nonzero(bucket_rel <= REPORT_RELATIVE_TOLERANCE)}/{DOT_PRODUCT_COUNT} '    f'within {REPORT_RELATIVE_TOLERANCE:.1e}; max relative error = {bucket_rel.max():.6e}; '    f'bit-exact vs Baseline = {np.count_nonzero(bucket_bits == baseline_bits[final_cycles])}/{DOT_PRODUCT_COUNT}.'
)